In [ ]:
"""NLTK → 英文分词
spaCy → 英文分词、词性、命名实体识别等
jieba → 中文分词
HanLP → 中文 NLP
正则表达式 → 简单文本提取"""

In [8]:
import nltk
from nltk.tokenize import word_tokenize

nltk_data_path=r"D:\11\NLP\nltk_data"
if nltk_data_path not in nltk.data.path: nltk.data.path.insert(0,nltk_data_path)
print("NLTK搜索路径：",nltk.data.path)
punkt_ready=True
try: print("punkt_tab位置：",nltk.data.find("tokenizers/punkt_tab/english/"))
except LookupError: punkt_ready=False; print("缺少punkt_tab，请按上面的目录结构手动下载并解压。")

NLTK搜索路径： ['D:\\11\\NLP\\nltk_data', 'C:\\Users\\Administrator/nltk_data', 'C:\\Users\\Administrator\\.conda\\envs\\rl\\nltk_data', 'C:\\Users\\Administrator\\.conda\\envs\\rl\\share\\nltk_data', 'C:\\Users\\Administrator\\.conda\\envs\\rl\\lib\\nltk_data', 'C:\\Users\\Administrator\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data']
punkt_tab位置： D:\11\NLP\nltk_data\tokenizers\punkt_tab\english


In [9]:
text="Natural language processing is fascinating."
if punkt_ready:
    tokens=word_tokenize(text,language="english"); print(tokens)
else: print("资源未准备，跳过NLTK分词。")

['Natural', 'language', 'processing', 'is', 'fascinating', '.']


##  spaCy：从本地模型目录加载

In [10]:
import spacy
import os
spacy_model_path=r"D:\11\NLP\data\en_core_web_sm"
spacy_ready=os.path.isdir(spacy_model_path)
print("spaCy版本：",spacy.__version__); print("本地模型目录存在：",spacy_ready)

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


spaCy版本： 3.8.7
本地模型目录存在： True


In [11]:
import en_core_web_sm

nlp=en_core_web_sm.load()#把英文模型加载到内存里面。
doc=nlp("Apple is looking at buying a U.K. startup for $1 billion.")
'''看起来只有一句代码。但它内部实际上已经完成了：
原始英文
   ↓
Tokenizer
   ↓
分成 Token
   ↓
Token变成模型特征
   ↓
词性标注
   ↓
依存句法分析
   ↓
属性规则处理
   ↓
词形还原
   ↓
命名实体识别
   ↓
所有结果保存到 doc'''
print("处理管道：",nlp.pipe_names)
'''“处理管道”，可以理解成：一句话进入 spaCy 后，会依次经过多个 NLP 处理步骤，每一步负责一种任务，最后把所有分析结果都保存到 doc 里面。'''
print("Tokens：",[token.text for token in doc])
print("实体：",[(ent.text,ent.label_) for ent in doc.ents])

处理管道： ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Tokens： ['Apple', 'is', 'looking', 'at', 'buying', 'a', 'U.K.', 'startup', 'for', '$', '1', 'billion', '.']
实体： [('Apple', 'ORG'), ('U.K.', 'GPE'), ('$1 billion', 'MONEY')]


## 3. jieba：中文分词，不需要额外模型

In [12]:
import jieba
sentence="我爱自然语言处理"
print("精确模式：","/".join(jieba.cut(sentence,cut_all=False)))
print("全模式：","/".join(jieba.cut(sentence,cut_all=True)))
print("搜索模式：","/".join(jieba.cut_for_search(sentence)))

精确模式： 我/爱/自然语言/处理
全模式： 我/爱/自然/自然语言/语言/处理
搜索模式： 我/爱/自然/语言/自然语言/处理


### 可选：本地自定义词典

In [13]:
userdict_path=r"D:\11\NLP\data\userdict.txt"
if os.path.isfile(userdict_path):
    jieba.load_userdict(userdict_path); print("已加载：",userdict_path)
else: print("没有自定义词典，继续使用jieba默认词典。")

没有自定义词典，继续使用jieba默认词典。


## HanLP：

In [ ]:
'''HanLP 可以理解成一个面向自然语言处理的工具/框架，尤其常用于中文 NLP。
它不只是能做分词。根据具体加载的模型不同，还可以用于：
中文分词
词性标注
命名实体识别
依存句法分析
语义分析等'''

In [14]:
try:
    import hanlp
    hanlp_installed=True
except ImportError:
    hanlp_installed=False
hanlp_model_path=r"D:\11\NLP\data\hanlp_model_local"
hanlp_ready=hanlp_installed and os.path.isdir(hanlp_model_path)
print("HanLP库已安装：",hanlp_installed); print("本地HanLP模型存在：",os.path.isdir(hanlp_model_path))

HanLP库已安装： True
本地HanLP模型存在： True


In [15]:
if hanlp_ready:
    hanlp_model=hanlp.load(hanlp_model_path); hanlp_result=hanlp_model("我爱自然语言处理"); print(hanlp_result)
else: print("HanLP本地模型未准备，跳过。")

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['我', '爱', '自然', '语言', '处理']


## 5. 综合中文处理流程：即使没有 spaCy/HanLP 也可运行

In [16]:
chinese_text="自然语言处理是人工智能的重要分支，近年来发展迅速。"
jieba_words=list(jieba.cut(chinese_text)); chinese_chars=re.findall(r"[\u4e00-\u9fff]+",chinese_text)
print("jieba分词：",jieba_words); print("正则提取连续中文：",chinese_chars)
if hanlp_ready: print("HanLP结果：",hanlp_model(chinese_text))
if spacy_ready: print("spaCy结果：",[(token.text,token.pos_) for token in nlp(chinese_text)])


jieba分词： ['自然语言', '处理', '是', '人工智能', '的', '重要', '分支', '，', '近年来', '发展', '迅速', '。']
正则提取连续中文： ['自然语言处理是人工智能的重要分支', '近年来发展迅速']
HanLP结果： ['自然', '语言', '处理', '是', '人工智能', '的', '重要', '分支', '，', '近年来', '发展', '迅速', '。']
spaCy结果： [('自然语言处理是人工智能的重要分支，近年来发展迅速', 'X'), ('。', 'ADP')]
